# PoliGraph model modernization

Run this notebook in Google Colab with a GPU runtime. Training data and outputs live in Google Drive, not Git. The notebook refuses to train without separate train/dev data and records versions and evaluation results with every artifact.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/poligraph-training')
NER = ROOT / 'ner'
PURPOSE = ROOT / 'purpose'
OUTPUT = ROOT / 'outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)
required_datasets = (
    NER / 'train.spacy',
    NER / 'dev.spacy',
    PURPOSE / 'train.jsonl',
    PURPOSE / 'test.jsonl',
)
missing_datasets = [path for path in required_datasets if not path.is_file()]
if missing_datasets:
    missing = '\n'.join(f'  - {path}' for path in missing_datasets)
    raise FileNotFoundError(
        'Training cannot start because these labeled datasets are missing:\n'
        f'{missing}\n\n'
        'Export independently labeled train/dev/test data into those exact Google Drive paths. '
        'Do not copy the production model predictions into the evaluation sets.'
    )
print('All required labeled datasets are present.')

In [ ]:
import subprocess
import sys
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > GPU before training.'
print(torch.cuda.get_device_name(0))
REPO = Path('/content/PoliGraph')
if (REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
elif REPO.exists():
    raise RuntimeError(f'{REPO} exists but is not a Git checkout; rename or remove it before retrying.')
else:
    subprocess.run(['git', 'clone', 'https://github.com/lukeblevins/PoliGraph.git', str(REPO)], check=True)
%cd /content/PoliGraph
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'models/colab-requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_md'], check=True)
with (OUTPUT / 'environment.txt').open('w') as environment:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], check=True, stdout=environment)

## Train and evaluate the privacy-policy NER model

`train.spacy` and `dev.spacy` must be independent datasets. Do not use the development set for training or promotion decisions become meaningless.

In [ ]:
subprocess.run([sys.executable, '-m', 'spacy', 'init', 'fill-config', 'models/named-entity-recognition/base_config.cfg', '/content/ner-config.cfg'], check=True)
subprocess.run([sys.executable, '-m', 'spacy', 'debug', 'data', '/content/ner-config.cfg', '--paths.train', str(NER / 'train.spacy'), '--paths.dev', str(NER / 'dev.spacy')], check=True)
subprocess.run([sys.executable, '-m', 'spacy', 'train', '/content/ner-config.cfg', '--gpu-id', '0', '--output', str(OUTPUT / 'ner-run'), '--paths.train', str(NER / 'train.spacy'), '--paths.dev', str(NER / 'dev.spacy')], check=True)
subprocess.run([sys.executable, '-m', 'spacy', 'evaluate', str(OUTPUT / 'ner-run/model-best'), str(NER / 'dev.spacy'), '--gpu-id', '0', '--output', str(OUTPUT / 'ner-metrics.json')], check=True)

## Train and evaluate purpose classification

This uses the fork's multi-label labels and the current SetFit API. Keep the held-out test file unchanged across candidate runs.

In [ ]:
subprocess.run([sys.executable, 'models/purpose-classification/train_setfit.py', str(PURPOSE / 'train.jsonl'), str(PURPOSE / 'test.jsonl'), str(OUTPUT / 'purpose-model'), '--metrics-output', str(OUTPUT / 'purpose-metrics.json')], check=True)

## Promotion gate

Review `ner-metrics.json`, `purpose-metrics.json`, and `environment.txt`. Promote only if each per-label metric and the macro score meet or exceed the recorded production baseline. After approval, package the two model directories as a versioned GitHub Release asset and update `fetch_data.py`; do not commit model binaries.